In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns

from sklearn.base import BaseEstimator , TransformerMixin
import torch as T
import torch.nn as nn
import torch.optim as O

from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from dataclasses import dataclass
from sklearn.model_selection import KFold
import tqdm
from torch.utils.data import TensorDataset , DataLoader



In [15]:
linux_path = r"/run/media/drdrakken/Elements/Sonstiges/Programmieren/Machine Learning/csvs/Covertype/covertype/covtype.data.gz"
column_names = [
    "Elevation",
    "Aspect",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points",
]

column_names += [f"Wilderness_Area_{i}" for i in range(1, 5)]
column_names += [f"Soil_Type_{i}" for i in range(1, 41)]
column_names += ["Cover_Type"]

df = pd.read_csv(
    linux_path,
    header=None,
    names=column_names
)

In [16]:
df

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,...,Soil_Type_32,Soil_Type_33,Soil_Type_34,Soil_Type_35,Soil_Type_36,Soil_Type_37,Soil_Type_38,Soil_Type_39,Soil_Type_40,Cover_Type
0,2596,51,3,258,0,510,221,232,148,6279,...,0,0,0,0,0,0,0,0,0,5
1,2590,56,2,212,-6,390,220,235,151,6225,...,0,0,0,0,0,0,0,0,0,5
2,2804,139,9,268,65,3180,234,238,135,6121,...,0,0,0,0,0,0,0,0,0,2
3,2785,155,18,242,118,3090,238,238,122,6211,...,0,0,0,0,0,0,0,0,0,2
4,2595,45,2,153,-1,391,220,234,150,6172,...,0,0,0,0,0,0,0,0,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581007,2396,153,20,85,17,108,240,237,118,837,...,0,0,0,0,0,0,0,0,0,3
581008,2391,152,19,67,12,95,240,237,119,845,...,0,0,0,0,0,0,0,0,0,3
581009,2386,159,17,60,7,90,236,241,130,854,...,0,0,0,0,0,0,0,0,0,3
581010,2384,170,15,60,5,90,230,245,143,864,...,0,0,0,0,0,0,0,0,0,3


In [ ]:
@dataclass
class Config():
    test_size: int = 0.2
    seed : int = 1234
    target : str = "Cover_Type"

    
config = Config()


In [18]:
class DataClass():
    def __init__(self):
        self.data = df.copy()
        config = Config()
        self.x = self.data.drop([config.target] , axis = 1)
        self.y = self.data[config.target]

        self.numerical_data = self.x.select_dtypes(include = "number").columns
        self.categorical_data = self.x.select_dtypes(exclude = "number").columns


In [19]:
class Visualize():

    def __init__(self):
        self.data = df.copy()

    def iqr(self , TargetCol):
        q1 = self.data[TargetCol].quantile(0.25)
        q3 = self.data[TargetCol].quantile(0.75)
        iqr = q3 - q1
        return q1 , q3 , iqr
    
    def data_skew(self , Skew_Col):
        data_skew = self.data[Skew_Col].skew()
        print(f"Skew Of Col:{Skew_Col}:-> :{data_skew}")

    def plot(self):
        for vis in self.data.numerical_data:
            fig , axes = plt.subplots(3 , 1 , figsize = (10 , 10) , dpi = 200)
            q1 , q3 , _ = self.iqr(TargetCol = vis)
            mean = self.data[vis].mean()
            self.data_skew(Skew_Col = vis)

            sns.histplot(data = self.data , x = vis, ax = axes[0])
            axes[0].axvline(q1 , color = "green")
            axes[0].axvline(q3 , color = "red")
            axes[0].axvline(mean , color = "yellow")
            axes[0].set_title(f"Hitsplot for :{vis}")

            sns.boxplot(data = self.data , x = vis, ax = axes[1])
            axes[1].axvline(q1 , color = "green")
            axes[1].axvline(q3 , color = "red")
            axes[1].axvline(mean , color = "yellow")
            axes[1].set_title(f"boxplot for :{vis}")

            sns.scatterplot(data = self.data , x=vis, y=self.data[vis] , ax = axes[2])
            axes[2].axvline(q1 , color = "green")
            axes[2].axvline(q3 , color = "red")
            axes[2].axvline(mean , color = "yellow")
            axes[2].set_title(f"scatterplot for :{vis}")

            plt.tight_layout()
            plt.show()

In [21]:
class DataTransform(BaseEstimator , TransformerMixin):

    def fit(self, X , y = None):
        return self
    
    def transform(self , X , y = None):
        x = self.new_features(X)
        x = self.combine_feature(x)
        return x
    
    def new_features(self, X):
        x = X.copy()
        x["DistanceHodrology"] = x["Horizontal_Distance_To_Hydrology"] + x["Vertical_Distance_To_Hydrology"] + x["Horizontal_Distance_To_Roadways"]
        x["Hillshade"] = x["Hillshade_9am"] + x["Hillshade_Noon"] + x["Hillshade_3pm"]
        return x
    
    def combine_feature(self , X):
        x = X.copy()
        features = [cols for cols in x.columns if cols.startswith("Soil_Type")]
        x["SoilType"] = x[features].values.argmax(axis = 1) + 1
        return x

In [ ]:
def data_split(data = None ):
    X_train , X_test , y_train , y_test = train_test_split(data.x,
                                                           data.y,
                                                           shuffle = True,
                                                           random_state = config.seed,
                                                           stratify = data.y)
    
    return X_train , X_test , y_train , y_test

In [ ]:
class Preprocessor():

    def fit(self , X , y = None):
        numerical_data = X.select_dtypes(inluce = np.number).columns
        categorical_data = X.select_dtypes(inluce = np.number).columns

        self.preproces_data = ColumnTransformer([
            ("numerical_preprocess" , Pipeline([
                ("DataTransformation" , DataTransform()),
                ("imputer" , SimpleImputer(strategy = "mean")),
                ("scale" , MinMaxScaler()),
            ]),numerical_data),

            ("categorical_preprocess" , Pipeline([
                ("imputer" , SimpleImputer(strategy = "most_frequent")),
                ("encoder" , OneHotEncoder(handle_unknown = "ignore")),
            ]),categorical_data),
        ])
        
        self.preproces_data.fit(X)
        return self
    
    def fit_transform(self, X ):
        return self.fit(X).transform(X)


In [ ]:
class NN(nn.Module):
    def __init__(self , InDims, OutDims):
        super().__init__()
        self.ll1 = nn.Linear(InDims, 128)
        self.ll2 = nn.Linear(128, 64)
        self.ll3 = nn.Linear(64, OutDims)
        self.activation = nn.ReLU()
        self.droprate = 0.2
        self.drop = nn.Dropout(self.droprate)

    def forward(self, X):
        x = self.activation(self.ll1(X))
        x = self.activation(self.ll2(x))
        x = self.ll3(x)

        return x

In [25]:
class Training():

    def __init__(self):
        self.X_train , self.X_test , self.y_train , self.y_test = data_split()
        
        self.epochs = 1
        self.device = T.device("cuda:0" if T.cuda.is_available() else "cpu")
        self.loss = nn.CrossEntropyLoss()
        self.lr = 1e-3
        self.batch_size = 32
        self.average_training_loss = []
        self.average_test_loss = []

        self.preprocessor = Preprocessor()
        
        self.X_train = self.preprocessor.fit_transform(self.X_train)
        self.X_test = self.preprocessor.transform(self.X_test)

        self.X_train = T.tensor(self.X_train , dtype = T.float32).to(self.device)
        self.X_test = T.tensor(self.X_test , dtype = T.float32).to(self.device)

        self.y_train = T.tensor(self.y_train.values , dtype = T.long).to(self.device) -1
        self.y_test = T.tensor(self.y_test.values , dtype = T.long).to(self.device) - 1

        self.model = NN(InDims= self.X_train.shape[1] , OutDims=len(T.unique(self.y_train))).to(self.device)
        self.optim = O.Adam(self.model.parameters() , lr = self.lr)

        self.train_data = TensorDataset(self.X_train , self.y_train)
        self.test_data = TensorDataset(self.X_test , self.y_test)

        self.train_loader = DataLoader(dataset = self.train_data,
                                       batch_size = self.batch_size,
                                       shuffle = True)
        
        self.test_loader = DataLoader(dataset = self.test_data,
                                       batch_size = self.batch_size)

        self.train_iterations()
        self.evaluation()

    def train_iterations(self):
            self.model.train()
            for epochs in range(self.epochs):
                current_loss = 0.0
                with tqdm.tqdm(iterable=self.train_loader , mininterval = 0.1 , disable = False , desc= f"Epoch:{epochs + 1}") as loading_bar:
                    for X , y in loading_bar:
                        X , y = X.to(self.device) , y.to(self.device)

                        self.optim.zero_grad()
                        logits = self.model(X)
                        loss = self.loss(logits , y)
                        loss.backward()
                        self.optim.step()

                        current_loss += loss.item()
                        loading_bar.set_postfix({"Loss:" : loss.item()})

                self.average_training_loss.append(current_loss / len(self.train_loader))
                print(f"Epoch:{epochs + 1} / {self.epochs} | Loss:{current_loss / len(self.train_loader)}")

    def evaluation(self):
        self.model.eval()
        current_loss = 0.0
        with T.no_grad():
            for epochs in range(self.epochs):
                with tqdm.tqdm(iterable=self.test_loader , mininterval = 0.1 , disable = False , desc= f"Epoch:{epochs + 1}") as loading_bar:
                    for X , y in loading_bar:
                        X , y = X.to(self.device) , y.to(self.device)

                        logits = self.model(X)
                        loss = self.loss(logits , y)
        
                        current_loss += loss.item()
                        loading_bar.set_postfix({"Loss:" : loss.item()})

                self.average_test_loss.append(current_loss / len(self.train_loader))
                print(f"Epoch:{epochs + 1} / {self.epochs} | Loss:{current_loss / len(self.train_loader)}")

In [26]:
Training()

Epoch:1: 100%|██████████| 13618/13618 [00:27<00:00, 498.99it/s, Loss:=0.523]


Epoch:1 / 1 | Loss:0.6233954730691635


Epoch:1: 100%|██████████| 4540/4540 [00:04<00:00, 1071.07it/s, Loss:=0.514]

Epoch:1 / 1 | Loss:0.18307366448210136
